This notebook demonstrates a simple **multimodal retrieval-augmented generation (RAG)** pipeline:

1. Initialize Vertex AI
2. Run a small Gemini text sanity check
3. Run a small image understanding sanity check
4. Create image embeddings for a small product catalog
5. Store those embeddings in **Astra DB**
6. Query the vector store with a new image
7. Use Gemini to generate a grounded final answer using the retrieved products

**Use case:** identify a coffee machine part from an image and suggest similar replacement products.

Install Vertex AI SDK

In [1]:
!pip install --upgrade --user google-cloud-aiplatform

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 39.0 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
!pip install ragstack-ai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.7/86.7 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 20.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 4.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of llama-index-indices-managed-llama-cloud to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-index-indices-managed-llama-cloud to determine which version is compatible with other requirements. This could take a while

In [1]:
import getpass, os, requests

if "GCP_PROJECT_ID" not in os.environ:
  os.environ["GCP_PROJECT_ID"] = getpass.getpass("Provide your GCP Project ID")

if "LOCATION" not in os.environ:
  os.environ["LOCATION"] = getpass.getpass("Provide your GCP Location")

if "ASTRA_DB_ENDPOINT" not in os.environ:
  os.environ["ASTRA_DB_ENDPOINT"] = getpass.getpass("Provide your Astra DB Endpoint")

if "ASTRA_DB_TOKEN" not in os.environ:
  os.environ["ASTRA_DB_TOKEN"] = getpass.getpass("Provide your Astra DB Token")

Provide your GCP Project ID··········
Provide your GCP Location··········
Provide your Astra DB Endpoint··········
Provide your Astra DB Token··········


Authenticate your notebook environment

In [2]:
!gcloud config set project {os.getenv("GCP_PROJECT_ID")}

Updated property [core/project].


In [3]:
import sys

# Additional authentication is required for Google Colab
if "google.colab" in sys.modules:
    # Authenticate user to Google Cloud
    from google.colab import auth

    auth.authenticate_user()

In [4]:
!gcloud auth list

     Credentialed Accounts
ACTIVE  ACCOUNT
*       er.tridibghosh@gmail.com

To set the active account, run:
    $ gcloud config set account `ACCOUNT`



Configure Google cloud project information and initialize Vertex AI SDK

In [5]:
# Define project information
PROJECT_ID=os.getenv("GCP_PROJECT_ID")
LOCATION=os.getenv("LOCATION")

In [7]:
#initialize VertexAI
import vertexai

vertexai.init(project=PROJECT_ID, location=LOCATION)

Import libraries

In [8]:
from vertexai.preview.generative_models import (
    GenerationConfig,
    GenerativeModel,
    HarmCategory,
    HarmBlockThreshold,
    Image,
    Part
)

Load the Gemini 2.5 model

In [77]:
model = GenerativeModel("gemini-2.5-pro")

/root/.local/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()



Test -- model is generating text correctly from text only prompts

In [13]:
responses = model.generate_content("Why is the sky blue?", stream=True)
for response in responses:
    print(response.text, end="")

Of course! This is a classic and wonderful question.

The short answer is that the sky is blue because of how the Earth's atmosphere scatters sunlight.

For a more detailed explanation, let's break it down into a few key steps.

### 1. Sunlight Isn't Just "White"
Sunlight looks white to us, but it's actually made up of all the colors of the rainbow: red, orange, yellow, green, blue, indigo, and violet (ROYGBIV). Each of these colors travels in a wave, and each has a different wavelength. Red has the longest wavelength, and blue and violet have the shortest.



### 2. Earth's Atmosphere is a Filter
Our planet is wrapped in a blanket of gases and tiny particles called the atmosphere. It's mostly made of nitrogen and oxygen molecules. As sunlight travels from the Sun to your eyes, it has to pass through this atmosphere.

### 3. The Key: Rayleigh Scattering
When sunlight hits the tiny gas molecules in the atmosphere, it gets scattered, meaning it bounces off in all different directions.

T

In [14]:
prompt = """Create a numbered list of 10 items. Each item in the list should be a trend in the tech industry.

Each trend should be less than 5 words."""

responses = model.generate_content(prompt, stream=True)
for response in responses:
    print(response.text, end="")

1.  Generative AI
2.  Edge Computing
3.  Zero Trust Security
4.  Sustainable Technology
5.  Low-Code/No-Code Platforms
6.  Virtual & Augmented Reality
7.  Applied Machine Learning
8.  Hybrid Cloud Adoption
9.  Web3 and Decentralization
10. Robotic Process Automation

**Model parameters**
Every prompt you send to the model includes parameter values that control how the model generates a response. The model can generate different results for different parameter values. You can experiment with different model parameters to see how the results change.

In [15]:
generation_config = GenerationConfig(
    temperature=0.9,
    top_p=1.0,
    top_k=32,
    candidate_count=1,
    max_output_tokens=8192,
)

In [17]:
#test
responses = model.generate_content(
    "Why is the sky blue?",
    generation_config=generation_config,
    stream=True,
)
for response in responses:
    print(response.text, end="")

Of course! This is one of the most classic and interesting science questions.

The short answer is: **The sky is blue because the Earth's atmosphere scatters blue light from the sun more than it scatters other colors.**

Here’s a more detailed, step-by-step explanation:

### 1. Sunlight Isn't Just One Color

The light from our sun, which appears white to us, is actually a mixture of all the colors of the rainbow. You can see this when light passes through a prism (or in a rainbow). Each color has a different wavelength:
*   **Red** has the longest wavelength.
*   **Violet** and **Blue** have the shortest wavelengths.

### 2. The Earth's Atmosphere is in the Way

Our atmosphere is a layer of gases, mostly nitrogen and oxygen. As sunlight travels through the atmosphere to reach our eyes, it collides with these tiny gas molecules.

### 3. The Scattering Effect (Rayleigh Scattering)

When sunlight hits these tiny gas molecules, the light gets scattered, meaning it's absorbed and then sent 

**Small multimodal sanity check**

Before adding retrieval, we first test whether Gemini can accept a local uploaded image and answer a simple question about it.

In [73]:
source_img_path = "coffee_maker_part.jpg"

In [74]:
from vertexai.generative_models import GenerativeModel, Part, Image
from PIL import Image as PILImage, ImageFile
import os, sys

In [75]:
source_img_path = "coffee_maker_part.jpg"

text_prompt = "What is this image? Share a link to purchase a replacement."

In [78]:
image_part = Part.from_image(Image.load_from_file(source_img_path))

response = model.generate_content(
    [text_prompt, image_part],
    safety_settings={
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_LOW_AND_ABOVE
    }
)

print(response.text)

Of course! Here's information about the item in your image and a link to purchase a replacement.

### What is this image?

This is a **non-pressurized espresso filter basket**.

It's a crucial component of an espresso machine. This metal basket fits inside the portafilter (the handled device you lock into the machine) and holds the ground coffee. During the brewing process, hot, pressurized water is forced through the coffee grounds and out through the tiny holes in the bottom of this basket to create espresso.

### How to Purchase a Replacement

Finding the correct replacement is critical, as filter baskets are **not one-size-fits-all**. You must match the basket's diameter to your espresso machine's portafilter. Common sizes are 51mm, 54mm, and 58mm.

To find the right one, you need to know the specific make and model of your espresso machine.

Here is a general search link on Amazon where you can find various sizes. Be sure to select the correct diameter for your machine:

**[Search

**Create a small sample product catalog**

For simplicity, this notebook uses a small in-memory catalog of coffee machine accessories and replacement parts.
Each row contains product metadata and an image URL.

In [39]:
import pandas as pd

d = {'name': ["Saucer", "Saucer Ceramic", "Milk Jug Assembly", "Handle Steam Wand Kit (New Version From 0735 PDC)", "Spout Juice Small (From 0637 to 1041 PDC)", "Cleaning Steam Wand", "Jug Frothing", "Spoon Tamping 50mm", "Collar Grouphead 50mm", "Filter 2 Cup Dual Wall 50mm", "Filter 1 Cup 50mm", "Water Tank Assembly", "Portafilter Assembly 50mm", "Milk Jug Assembly", "Filter 2 Cup 50mm" ],
     'url': ["https://www.breville.com/us/en/parts-accessories/parts/sp0014946.html?sku=SP0014946", "https://www.breville.com/us/en/parts-accessories/parts/sp0014914.html?sku=SP0014914", "https://www.breville.com/us/en/parts-accessories/parts/sp0011391.html?sku=SP0011391", "https://www.breville.com/us/en/parts-accessories/parts/sp0010719.html?sku=SP0010719", "https://www.breville.com/us/en/parts-accessories/parts/sp0010718.html?sku=SP0010718", "https://www.breville.com/us/en/parts-accessories/parts/sp0003247.html?sku=SP0003247", "https://www.breville.com/us/en/parts-accessories/parts/sp0003246.html?sku=SP0003246", "https://www.breville.com/us/en/parts-accessories/parts/sp0003243.html?sku=SP0003243", "https://www.breville.com/us/en/parts-accessories/parts/sp0003232.html?sku=SP0003232", "https://www.breville.com/us/en/parts-accessories/parts/sp0003231.html?sku=SP0003231", "https://www.breville.com/us/en/parts-accessories/parts/sp0003230.html?sku=SP0003230", "https://www.breville.com/us/en/parts-accessories/parts/sp0003225.html?sku=SP0003225", "https://www.breville.com/us/en/parts-accessories/parts/sp0003216.html?sku=SP0003216", "https://www.breville.com/us/en/parts-accessories/parts/sp0001875.html?sku=SP0001875", "https://www.breville.com/us/en/parts-accessories/parts/sp0000166.html?sku=SP0000166"],
     'price': ["10.95", "4.99", "14.95", "8.95", "10.95", "6.95", "24.95", "8.95", "6.95", "12.95", "12.95", "14.95", "10.95", "16.95", "11.95"],
     'image': ["https://www.breville.com/content/dam/breville/us/catalog/products/images/sp0/sp0014946/tile.jpg", "https://www.breville.com/content/dam/breville/us/catalog/products/images/sp0/sp0014914/tile.jpg", "https://www.breville.com/content/dam/breville/us/catalog/products/images/sp0/sp0011391/tile.jpg", "https://www.breville.com/content/dam/breville/ca/catalog/products/images/sp0/sp0010719/tile.jpg", "https://www.breville.com/content/dam/breville/ca/catalog/products/images/sp0/sp0010718/tile.jpg", "https://www.breville.com/content/dam/breville/ca/catalog/products/images/sp0/sp0003247/tile.jpg", "https://assets.breville.com/cdn-cgi/image/width=400,format=auto/Spare+Parts+/Espresso+Machines/BES250/SP0003246/SP0003246_IMAGE1_400X400.jpg", "https://assets.breville.com/cdn-cgi/image/width=400,format=auto/Spare+Parts+/Espresso+Machines/ESP8/SP0003243/SP0003243_IMAGE1_400X400.jpg", "https://assets.breville.com/cdn-cgi/image/width=400,format=auto/Spare+Parts+/Espresso+Machines/ESP8/SP0003232/SP0003232_IMAGE1_400x400.jpg", "https://www.breville.com/content/dam/breville/au/catalog/products/images/sp0/sp0003231/tile.jpg", "https://www.breville.com/content/dam/breville/au/catalog/products/images/sp0/sp0003230/tile.jpg", "https://www.breville.com/content/dam/breville/ca/catalog/products/images/sp0/sp0003225/tile.jpg", "https://www.breville.com/content/dam/breville/ca/catalog/products/images/sp0/sp0003216/tile.jpg", "https://www.breville.com/content/dam/breville/au/catalog/products/images/sp0/sp0001875/tile.jpg", "https://www.breville.com/content/dam/breville/us/catalog/products/images/sp0/sp0000166/tile.jpg"]}
df = pd.DataFrame(data=d)
df

,name,url,price,image
0,Saucer,https://www.breville.com/us/en/parts-accessori...,10.95,https://www.breville.com/content/dam/breville/...
1,Saucer Ceramic,https://www.breville.com/us/en/parts-accessori...,4.99,https://www.breville.com/content/dam/breville/...
2,Milk Jug Assembly,https://www.breville.com/us/en/parts-accessori...,14.95,https://www.breville.com/content/dam/breville/...
3,Handle Steam Wand Kit (New Version From 0735 PDC),https://www.breville.com/us/en/parts-accessori...,8.95,https://www.breville.com/content/dam/breville/...
4,Spout Juice Small (From 0637 to 1041 PDC),https://www.breville.com/us/en/parts-accessori...,10.95,https://www.breville.com/content/dam/breville/...
5,Cleaning Steam Wand,https://www.breville.com/us/en/parts-accessori...,6.95,https://www.breville.com/content/dam/breville/...
6,Jug Frothing,https://www.breville.com/us/en/parts-accessori...,24.95,https://assets.breville.com/cdn-cgi/image/widt...
7,Spoon Tamping 50mm,https://www.breville.com/us/en/parts-accessori...,8.95,https://assets.breville.com/cdn-cgi/image/widt...
8,Collar Grouphead 50mm,https://www.breville.com/us/en/parts-accessori...,6.95,https://assets.breville.com/cdn-cgi/image/widt...
9,Filter 2 Cup Dual Wall 50mm,https://www.breville.com/us/en/parts-accessori...,12.95,https://www.breville.com/content/dam/breville/...


Load the multimodal embedding model and Astra DB client.

The embedding model converts each catalog image into a dense vector.
Astra DB stores those vectors so we can perform similarity search later.

In [40]:
import vertexai, json, requests
from vertexai.preview.vision_models import MultiModalEmbeddingModel, Image
from astrapy.db import AstraDB, AstraDBCollection
from google.colab import files

/tmp/ipykernel_15467/358620499.py:3: DeprecatedWarning: All of 'astrapy.core.*', 'astrapy.api.*', 'astrapy.db.*' and 'astrapy.ops.*' is deprecated as of 1.5.0 and will be removed in 2.0.0. Please refer to https://docs.datastax.com/en/astra-db-serverless/api-reference/dataapiclient.html to start the (recommended) path forward to the current API, and to https://github.com/datastax/astrapy?tab=readme-ov-file#appendix-b-compatibility-with-pre-100-library for details on the deprecated modules.
  from astrapy.db import AstraDB, AstraDBCollection


In [41]:
model = MultiModalEmbeddingModel.from_pretrained("multimodalembedding@001")

/root/.local/lib/python3.12/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [43]:
# Initialize our vector db
astra_db = AstraDB(token=os.getenv("ASTRA_DB_TOKEN"), api_endpoint=os.getenv("ASTRA_DB_ENDPOINT"))

Create a vector collection

We create a collection with dimension **1408**, which matches the output size of the multimodal embedding model.

In [44]:
collection = astra_db.create_collection(collection_name="coffee_shop_ecommerce", dimension=1408)

Index the catalog into Astra DB

For each product:
- download the product image
- compute its image embedding
- store the product metadata and vector in Astra DB

In [45]:
for i in range(len(df)):
  name = df.loc[i, "name"]
  image = df.loc[i, "image"]
  price = df.loc[i, "price"]
  url = df.loc[i, "url"]

  # Download this product's image and save it to the Colab filesystem.
  # In a production system this binary data would be stored in Google Cloud Storage
  img_data = requests.get(image).content
  with open(f'{name}.png', 'wb') as handler:
    handler.write(img_data)

  # load the image from filesystem and compute the embedding value
  img = Image.load_from_file(f'{name}.png')
  embeddings = model.get_embeddings(image=img, contextual_text=name)

  try:
    # add to the AstraDB Vector Database
    collection.insert_one({
        "_id": i,
        "name": name,
        "image": image,
        "url": url,
        "price": price,
        "$vector": embeddings.image_embedding,
      })
  except Exception as error:
    # if you've already added this record, skip the error message
    error_info = json.loads(str(error))
    if error_info[0]['errorCode'] == "DOCUMENT_ALREADY_EXISTS":
      print("Document already exists in the database.  Skipping.")

/root/.local/lib/python3.12/site-packages/vertexai/vision_models/_vision_models.py:154: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Embed the query image

Now we embed the uploaded test image that we want to identify.

In [47]:
import json

# Embed the similar item
img = Image.load_from_file('coffee_maker_part.jpg')

In [48]:
embeddings = model.get_embeddings(image=img, contextual_text="A espresso machine part")

In [49]:
embeddings.image_embedding

[0.00638786331,
 -0.0420592204,
 0.00726634776,
 0.0434983335,
 -0.0111204181,
 -0.0362185873,
 0.00216504256,
 -0.0276085567,
 -0.0337016582,
 -0.0183864217,
 -0.0259998553,
 -0.0184864625,
 -0.00164292951,
 0.151931226,
 -0.0355522,
 0.00690923678,
 0.0542577468,
 0.00652649766,
 -0.0160229038,
 -0.00401567,
 -0.00758272968,
 0.00981625449,
 -0.0151233226,
 -0.0118413726,
 0.000865053269,
 0.0273249671,
 -0.00801409315,
 -0.00686218264,
 0.0631566271,
 -0.0280179791,
 0.0191293322,
 0.000176534377,
 0.0116350157,
 0.00020527601,
 0.0281073,
 -0.0131453201,
 -0.0137607912,
 0.0168333482,
 -0.00713033369,
 -0.0117478091,
 -0.0228877496,
 0.00151367055,
 -0.0150919808,
 -0.0319481753,
 0.0224684123,
 -0.00690934947,
 -0.00557175139,
 -0.0197909474,
 0.0207019653,
 -0.00965946,
 -0.0256833266,
 -0.0183929503,
 -0.00130401656,
 -0.022131566,
 0.0373204723,
 -0.00564531097,
 -0.0354033895,
 -0.0162405092,
 0.029561121,
 -0.0257030949,
 0.0233057514,
 -0.018611297,
 0.0199442301,
 -0.016523

Retrieve similar products from Astra DB

We perform vector similarity search using the query image embedding and fetch the top matching products.

In [50]:
# Perform the vector search against AstraDB Vector

documents = collection.vector_find(
    embeddings.image_embedding,
    limit=3,
)

In [51]:
documents

[{'_id': 10,
  'name': 'Filter 1 Cup 50mm',
  'image': 'https://www.breville.com/content/dam/breville/au/catalog/products/images/sp0/sp0003230/tile.jpg',
  'url': 'https://www.breville.com/us/en/parts-accessories/parts/sp0003230.html?sku=SP0003230',
  'price': '12.95',
  '$similarity': 0.941376},
 {'_id': 14,
  'name': 'Filter 2 Cup 50mm',
  'image': 'https://www.breville.com/content/dam/breville/us/catalog/products/images/sp0/sp0000166/tile.jpg',
  'url': 'https://www.breville.com/us/en/parts-accessories/parts/sp0000166.html?sku=SP0000166',
  'price': '11.95',
  '$similarity': 0.9397586},
 {'_id': 9,
  'name': 'Filter 2 Cup Dual Wall 50mm',
  'image': 'https://www.breville.com/content/dam/breville/au/catalog/products/images/sp0/sp0003231/tile.jpg',
  'url': 'https://www.breville.com/us/en/parts-accessories/parts/sp0003231.html?sku=SP0003231',
  'price': '12.95',
  '$similarity': 0.9397576}]

Convert retrieved results into prompt context

The retrieved products are formatted into a compact CSV-like text block that will be passed to Gemini as grounded context.

In [52]:
related_products_csv = "name, image, price, url\n"
for doc in documents:
  related_products_csv += f"{doc['name']}, {doc['image']}, {doc['price']}, {doc['url']},\n"

In [53]:
print(related_products_csv)

name, image, price, url
Filter 1 Cup 50mm, https://www.breville.com/content/dam/breville/au/catalog/products/images/sp0/sp0003230/tile.jpg, 12.95, https://www.breville.com/us/en/parts-accessories/parts/sp0003230.html?sku=SP0003230,
Filter 2 Cup 50mm, https://www.breville.com/content/dam/breville/us/catalog/products/images/sp0/sp0000166/tile.jpg, 11.95, https://www.breville.com/us/en/parts-accessories/parts/sp0000166.html?sku=SP0000166,
Filter 2 Cup Dual Wall 50mm, https://www.breville.com/content/dam/breville/au/catalog/products/images/sp0/sp0003231/tile.jpg, 12.95, https://www.breville.com/us/en/parts-accessories/parts/sp0003231.html?sku=SP0003231,



Final grounded multimodal answer

In the final step, we combine:
- the **query image**
- the **retrieved product context**
- a **natural-language instruction**

Gemini then generates a grounded answer describing the part and suggesting similar replacements.

In [79]:
source_img_path = "coffee_maker_part.jpg"

text_prompt = f"""
What do we have in this image?
Share the product name, and if possible suggest a replacement URL and price.

Here are related products:
{related_products_csv}
"""

In [80]:
image_part = Part.from_image(Image.load_from_file(source_img_path))

response = model.generate_content(
    [text_prompt, image_part],
    safety_settings={
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_LOW_AND_ABOVE
    }
)

print(response.text)

Based on the image provided and the related products, here is the information for the item in the picture:

**Product Name:** Filter 2 Cup 50mm

**Replacement URL:** https://www.breville.com/us/en/parts-accessories/parts/sp0000166.html?sku=SP0000166

**Price:** $11.95
